| Roll No.| Subject Code | Assignment No. |
|:-------:|:------------:|:--------------:|
| 2650026 | CS69101      | Lab 4          |

Given a set of integer elements stored in an input file, `input.txt`, construct a Red-Black Tree by inserting the elements one by one. Implement insertion and deletion operations while maintaining all the Red-Black Tree properties. Store the final tree contents in `output.txt` using in-order traversal.

In [1]:
import sys
from io import TextIOWrapper
from typing import Self
import random
import sys
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pylab import figure

sys.setrecursionlimit(100000) # Fix to avoid RecursionError

prep_arr = [random.randint(1, (i+1)*10) for i in range(5000)] # Random Array of 5000 Elements
# prep_arr.sort() # Sorted Array for Worst Case Scenario
print(prep_arr[:25])
print(len(prep_arr))

# Write the array to a file
f = open("./2650026_lab4_data/input0.txt", "w")
for i in prep_arr:
    f.write(f"{i}, ")
f.write("\n")
f.close()

del prep_arr
del f

iterations = 0
PRINT_DEBUG = False

[7, 7, 5, 30, 46, 58, 45, 60, 77, 16, 16, 5, 128, 16, 23, 137, 133, 87, 156, 127, 98, 206, 58, 90, 136]
5000


# Q1 & Q2

Read integer contents from `input.txt` and construct a Red-Black Tree using the `insert(key)` operation.

Implement Red-Black Tree insertion, including the required recolouring and LL, RR, LR, and RL rotations to maintain the Red-Black Tree properties.

## File Read

In [2]:
# Read a set of integer elements from an input file (input.txt)
arr0 = []
data0 = open("./2650026_lab4_data/input.txt", 'r')

try:
    for line in data0.readlines():
        line_data = line.rstrip().rstrip(',').replace(' ','').split(',') #using rstrip to remove the \n
        for data in line_data:
            arr0.append(int(data))
    data0.close()
except Exception as e:
    sys.exit("[ERROR] Invalid data" + str(e))

In [3]:
print(arr0)

[70, 50, 90, 77, 55, 348, 76, 67, 69, 420, 25, 911, 210, 108, 768, 1080, 720]


## Red Black Tree

In [4]:
class RB_Node:
    def __init__(self, value: int | None = None) -> None:
        self.__value = value
        self.__lchild = None
        self.__rchild = None
        self.__parent = None
        self.__color = '#F00' # Color in #RGB, new node is always red

    def __get_lchild(self) -> Self | None:
        return self.__lchild
    
    def __set_lchild(self, lchild: Self | None) -> None: # type: ignore
        if PRINT_DEBUG:
            print(f"Setting left child of node with value {self.__value} to {lchild}")
        self.__lchild = lchild
        if lchild is not None:
            lchild.__parent = self
            # lchild.update_height_and_balance()  # Update the height and balance factor of the left child when it is set
        # self.update_height_and_balance()  # Update the height and balance factor of the current node when the left child is set
    
    lchild = property(__get_lchild, __set_lchild, doc="Left child of the node")
    
    def __get_rchild(self) -> Self | None:
        return self.__rchild
    
    def __set_rchild(self, rchild: Self | None) -> None: # type: ignore
        if PRINT_DEBUG:
            print(f"Setting right child of node with value {self.__value} to {rchild}")
        self.__rchild = rchild
        if rchild is not None:
            rchild.__parent = self
            # rchild.update_height_and_balance()  # Update the height and balance factor of the right child when it is set
        # self.update_height_and_balance()  # Update the height and balance factor of the current node when the right child is set
    
    rchild = property(__get_rchild, __set_rchild, doc="Right child of the node")
    
    def __get_parent(self) -> Self | None:
        return self.__parent
    
    def __set_parent(self, parent: Self | None) -> None: # type: ignore
        if PRINT_DEBUG:
            print(f"Setting parent of node with value {self.__value} to {parent}")
        self.__parent = parent
        if parent is not None:
            # parent.update_height_and_balance()  # Update the height and balance factor of the parent when it is set
            pass
    
    parent = property(__get_parent, __set_parent, doc="Parent of the node")
    
    def __get_value(self) -> int | None:
        return self.__value
    
    def __set_value(self, value: int) -> None:
        self.__value = value
    
    value = property(__get_value, __set_value, doc="Value of the node")

    def __get_color(self) -> str | None:
        return self.__color

    def __set_color(self, color: str) -> None:
        self.__color = color

    color = property(__get_color, __set_color, doc="Color of the node")
    
    def __str__(self) -> str:
        return f"Node(value={self.__value}, color={self.__color}, lchild={self.__lchild.value if self.__lchild else None}, rchild={self.__rchild.value if self.__rchild else None}, parent={self.__parent.value if self.__parent else None})"
    
    def __repr__(self) -> str:
        return self.__str__()

In [5]:
def RB_right_rotate(k2_node: RB_Node | None, root_node: RB_Node | None) -> RB_Node | None:
    if PRINT_DEBUG:
        print(f"Right Rotate: k2_node={k2_node}, root_node={root_node}")
    k1_node = k2_node.lchild if k2_node is not None else None
    if k1_node is None or k2_node is None:
        return root_node

    if PRINT_DEBUG:
        print(f"LL Case: k1_node={k1_node}, k2_node={k2_node}")

    # From Weiss' Data Structures and Algorithm Analysis in C++ (Page 156)
    # Combined CLRS Logic with Weiss' Logic to avoid bugs
    
    # SAFETY CODE
    TEMP_NODE = k1_node.rchild
    OG_PARENT = k2_node.parent
    WAS_RCHILD = bool(OG_PARENT is not None and OG_PARENT.rchild is k2_node)

    # CLRS: x.lchild = y.rchild
    # CLRS: y.rchild = x
    k1_node.rchild = k2_node
    k2_node.lchild = TEMP_NODE

    # Update Parents

    # CLRS: if y.rchild is not None:
    # CLRS:     y.rchild.parent = y
    # CLRS: // However, y.rchild is now x.lchild, so we need to update x.lchild's parent to y
    if PRINT_DEBUG:
        print(f"Updating parent of {k2_node.lchild} to {k2_node}")
    if k2_node.lchild is not None:
        k2_node.lchild.parent = k2_node

    # CLRS: x.parent = y.parent
    k1_node.parent = OG_PARENT

    # CLRS: if y.parent is None:
    # CLRS:     root_node = y
    if OG_PARENT is None:
        # root_node = k1_node
        pass # The input node will always be the root_node for the called function.
    # CLRS: elseif y is y.parent.rchild:
    # CLRS:     y.parent.rchild = x
    elif WAS_RCHILD:
        OG_PARENT.rchild = k1_node
        # OG_PARENT.update_height_and_balance() # Update height and balance factor of OG_PARENT
    # CLRS: else:
    # CLRS:     y.parent.lchild = x
    else:
        OG_PARENT.lchild = k1_node
        # OG_PARENT.update_height_and_balance() # Update height and balance factor of OG_PARENT

    root_node = k1_node # The input node will always be the root_node for the called function.

    # CLRS: y.parent = x
    # k2_node.parent = k1_node (THIS IS AUTO HANDLED BY SETTER METHOD)

    # if k1_node.rchild is not None:
    #     k1_node.rchild.parent = k1_node

    # root_node = k1_node if OG_PARENT is None else root_node // Copilot told me this

    # k1_node.update_height_and_balance() # Update height and balance factor of k1_node
    # k2_node.update_height_and_balance() # Update height and balance factor of k2_node

    if k1_node.parent is not None:
        # k1_node.parent.update_height_and_balance() # Update height and balance factor of k1_node's parent
        pass

    return root_node

def RB_left_rotate(k1_node: RB_Node | None, root_node: RB_Node | None) -> RB_Node | None:
    if PRINT_DEBUG:
        print(f"Left Rotate: k1_node={k1_node}, root_node={root_node}")
    k2_node = k1_node.rchild if k1_node is not None else None
    if k1_node is None or k2_node is None:
        return root_node

    if PRINT_DEBUG:
        print(f"RR Case: k1_node={k1_node}, k2_node={k2_node}")

    # From Weiss' Data Structures and Algorithm Analysis in C++ (Page 156)
    # Combined CLRS Logic with Weiss' Logic to avoid bugs
    
    # SAFETY CODE
    TEMP_NODE = k2_node.lchild
    OG_PARENT = k1_node.parent
    WAS_RCHILD = bool(OG_PARENT is not None and OG_PARENT.rchild is k1_node)

    # CLRS: x.rchild = y.lchild
    # CLRS: y.lchild = x
    k2_node.lchild = k1_node
    k1_node.rchild = TEMP_NODE

    # Update Parents

    # CLRS: if y.lchild is not None:
    # CLRS:     y.lchild.parent = x
    # CLRS: // However, y.lchild is now x.rchild, so we need to update x.rchild's parent to y
    if k1_node.rchild is not None:
        k1_node.rchild.parent = k1_node

    # CLRS: y.parent = x.parent
    k2_node.parent = OG_PARENT

    # CLRS: if x.parent is None:
    # CLRS:     root_node = y
    if OG_PARENT is None:
        # root_node = k2_node
        pass # The input node will always be the root_node for the called function.
    # CLRS: elseif x is x.parent.lchild:
    # CLRS:     x.parent.lchild = y
    elif WAS_RCHILD:
        OG_PARENT.rchild = k2_node
        # OG_PARENT.update_height_and_balance() # Update height and balance factor of OG_PARENT
    # CLRS: else:
    # CLRS:     x.parent.rchild = y
    else:
        OG_PARENT.lchild = k2_node
        # OG_PARENT.update_height_and_balance() # Update height and balance factor of OG_PARENT

    root_node = k2_node

    # CLRS: x.parent = y
    # k1_node.parent = k2_node (THIS IS AUTO HANDLED BY SETTER METHOD)
    
    # k1_node.update_height_and_balance() # Update height and balance factor of k1_node
    # k2_node.update_height_and_balance() # Update height and balance factor of k2_node

    if k2_node.parent is not None:
        # k2_node.parent.update_height_and_balance() # Update height and balance factor of k2_node's parent
        pass

    return root_node

#   k3
#   /
# k1          =>        k2
#  \                   / \
#   k2                k1 k3
    
def RB_double_rotate_right(k3_node: RB_Node | None, root_node: RB_Node | None) -> RB_Node | None:
    if PRINT_DEBUG:
        print(f"Double Right Rotate: k3_node={k3_node}, root_node={root_node}")
    k1_node = k3_node.lchild if k3_node is not None else None
    if k1_node is None or k3_node is None:
        return root_node

    if PRINT_DEBUG:
        print(f"Double Right Rotate: k1_node={k1_node}, k3_node={k3_node}")

    root_node = RB_left_rotate(k1_node, root_node)
    root_node = RB_right_rotate(k3_node, root_node)

    # k3_node.update_height_and_balance() # Update height and balance factor of k3_node
    # k1_node.update_height_and_balance() # Update height and balance factor of k1_node

    if k3_node.parent is not None:
        # k3_node.parent.update_height_and_balance() # Update height and balance factor of k3_node's parent
        pass

    return root_node

# k3
#   \
#   k1          =>      k2
#  /                   / \
# k2                  k1 k3

def RB_double_rotate_left(k3_node: RB_Node | None, root_node: RB_Node | None) -> RB_Node | None:
    if PRINT_DEBUG:
        print(f"Double Left Rotate: k3_node={k3_node}, root_node={root_node}")
    k1_node = k3_node.rchild if k3_node is not None else None
    if k1_node is None or k3_node is None:
        return root_node

    if PRINT_DEBUG:
        print(f"Double Left Rotate: k1_node={k1_node}, k3_node={k3_node}")

    root_node = RB_right_rotate(k1_node, root_node)
    root_node = RB_left_rotate(k3_node, root_node)

    # k3_node.update_height_and_balance() # Update height and balance factor of k3_node
    # k1_node.update_height_and_balance() # Update height and balance factor of k1_node

    if k3_node.parent is not None:
        # k3_node.parent.update_height_and_balance() # Update height and balance factor of k3_node's parent
        pass

    return root_node

In [6]:
def RB_Insert_Fixup(root: RB_Node, Z_Node: RB_Node) -> RB_Node | None:
    while Z_Node.parent is not None and Z_Node.parent.color == '#F00':
        if Z_Node.parent == Z_Node.parent.parent.lchild:
            Y_Node = Z_Node.parent.parent.rchild
            if Y_Node is not None and Y_Node.color == '#F00':
                Z_Node.parent.color = '#000'
                Y_Node.color = '#000'
                Z_Node.parent.parent.color = '#F00'
                Z_Node = Z_Node.parent.parent
            else:
                if Z_Node == Z_Node.parent.rchild:
                    Z_Node = Z_Node.parent
                    root = RB_left_rotate(Z_Node, root) # type: ignore
                Z_Node.parent.color = '#000'
                Z_Node.parent.parent.color = '#F00'
                root = RB_right_rotate(Z_Node.parent.parent, root) # type: ignore
        else:
            Y_Node = Z_Node.parent.parent.lchild
            if Y_Node is not None and Y_Node.color == '#F00':
                Z_Node.parent.color = '#000'
                Y_Node.color = '#000'
                Z_Node.parent.parent.color = '#F00'
                Z_Node = Z_Node.parent.parent
            else:
                if Z_Node == Z_Node.parent.lchild:
                    Z_Node = Z_Node.parent
                    root = RB_right_rotate(Z_Node, root) # type: ignore
                Z_Node.parent.color = '#000'
                Z_Node.parent.parent.color = '#F00'
                root = RB_left_rotate(Z_Node.parent.parent, root) # type: ignore
    root.color = '#000'
    return root

def RB_Insert(root: RB_Node | None, value: int) -> RB_Node | None:
    # global iterations
    # iterations += 1
    
    Z_Node = RB_Node(value)

    Y_NODE = None
    X_NODE = root
    while X_NODE is not None:
        Y_NODE = X_NODE
        if value < X_NODE.value:
            X_NODE = X_NODE.lchild
        else:
            X_NODE = X_NODE.rchild

    Z_Node.parent = Y_NODE

    if Y_NODE is None:
        root = Z_Node
    elif value < Y_NODE.value:
        Y_NODE.lchild = Z_Node
    else:
        Y_NODE.rchild = Z_Node

    Z_Node.lchild = None
    Z_Node.rchild = None
    # New Node's Color is already RED by default.

    root = RB_Insert_Fixup(root, Z_Node) # type: ignore
    return root


In [7]:
def RB_inorder_traversal(root: RB_Node | None, out_file: TextIOWrapper | None) -> None:
    if root is None:
        return
    RB_inorder_traversal(root.lchild, out_file = out_file)
    if out_file is not None:
        print(root.value, end = ', ', file = out_file)
    print(root.value, end = ', ')
    print(root)
    RB_inorder_traversal(root.rchild, out_file = out_file)

def RB_preorder_traversal(root: RB_Node | None, out_file: TextIOWrapper | None) -> None:
    if root is None:
        return
    if out_file is not None:
        print(root.value, end = ', ', file = out_file)
    print(root.value, end = ', ')
    print(root)
    RB_preorder_traversal(root.lchild, out_file)
    RB_preorder_traversal(root.rchild, out_file)

def RB_postorder_traversal(root: RB_Node | None, out_file: TextIOWrapper | None) -> None:
    if root is None:
        return
    RB_postorder_traversal(root.lchild, out_file)
    RB_postorder_traversal(root.rchild, out_file)
    if out_file is not None:
        print(root.value, end = ', ', file = out_file)
    print(root.value, end=', ')
    print(root)

In [8]:
root0 = None
for value in arr0:
    root0 = RB_Insert(root0, value)

In [ ]:
RB_inorder_traversal(root0, out_file = None)

25, Node(value=25, color=#F00, lchild=None, rchild=None, parent=67)
67, Node(value=67, color=#000, lchild=25, rchild=None, parent=69)
69, Node(value=69, color=#F00, lchild=67, rchild=210, parent=420)
108, Node(value=108, color=#F00, lchild=None, rchild=None, parent=210)
210, Node(value=210, color=#000, lchild=108, rchild=None, parent=69)
420, Node(value=420, color=#000, lchild=69, rchild=911, parent=55)
720, Node(value=720, color=#F00, lchild=None, rchild=None, parent=768)
768, Node(value=768, color=#000, lchild=720, rchild=None, parent=911)
911, Node(value=911, color=#F00, lchild=768, rchild=1080, parent=420)
1080, Node(value=1080, color=#000, lchild=None, rchild=None, parent=911)


# Q3.

Implement delete(key) to remove a specified element from the tree and perform the required recolouring and rotations/fix-up operations.